In [55]:
import torch
import torch.nn as nn

In [ ]:
## a simple NN with 5 neurons in input layer and 1 in target layer
class NeuralNetwork_v1(nn.Module):
    def __init__(self,features):
        super().__init__()
        self.linear=nn.Linear(features,1)
        self.sigmoid=nn.Sigmoid()

    def forward(self,inp):
        x=self.linear(inp)   
        x=self.sigmoid(x)
        return x 

In [ ]:
inp=torch.rand((10,5))
neural_net=NeuralNetwork_v1(5)
neural_net.forward(inp)

tensor([[0.4369],
        [0.4609],
        [0.4949],
        [0.5207],
        [0.4659],
        [0.4105],
        [0.4522],
        [0.4827],
        [0.4395],
        [0.4008]], grad_fn=<SigmoidBackward0>)

In [58]:
### using container
class NeuralNetwork_v2(nn.Module):
    def __init__(self,features):
        super().__init__()
        self.container=nn.Sequential(
            nn.Linear(features,3),
            nn.ReLU(),
            nn.Linear(3,1),
            nn.Sigmoid()
        )

    def forward(self,inp):
        out=self.container(inp) 
        return out 
    def loss(self,y_pred,target):
        return nn.CrossEntropyLoss(y_pred,target) 



In [59]:
neural_nets=NeuralNetwork_v2(5)
neural_nets.forward(inp)

tensor([[0.3777],
        [0.3882],
        [0.3901],
        [0.3777],
        [0.3828],
        [0.3792],
        [0.3859],
        [0.3938],
        [0.3777],
        [0.3817]], grad_fn=<SigmoidBackward0>)

In [61]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [62]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [63]:
df.drop(columns=['id', 'Unnamed: 32'], inplace= True)

In [64]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)


In [65]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [66]:
## label encoding
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [67]:
## torch typecasting
X_train_tensor = torch.from_numpy(X_train.astype(np.float32))
X_test_tensor = torch.from_numpy(X_test.astype(np.float32))
y_train_tensor = torch.from_numpy(y_train.astype(np.float32))
y_test_tensor = torch.from_numpy(y_test.astype(np.float32))

In [68]:
class NeuralNetwork_v3(nn.Module):
    def __init__(self,features):
        super().__init__()
        self.container=nn.Sequential(
            nn.Linear(features,3),
            nn.ReLU(),
            nn.Linear(3,1),
            nn.Sigmoid()
        )

    def forward(self,inp):
        out=self.container(inp) 
        return out

In [69]:
### Hyperparameter
LR=0.1
n_epochs=50

In [70]:
loss=nn.BCELoss()

In [71]:
y_train_tensor.shape

torch.Size([455])

#### Training pipeline


In [72]:
model=NeuralNetwork_v3(X_train_tensor.shape[1])
optimizer=torch.optim.SGD(model.parameters(),lr=LR)
for i in range(n_epochs):
    ## forward pass
    y_pred=model.forward(X_train_tensor)
    ## loss cal
    loss_cal=loss(y_pred,y_train_tensor.reshape(-1,1))
    optimizer.zero_grad()
    ### back pass
    loss_cal.backward()
    ### change param
    optimizer.step()
    if i%10==0:
        print(f"Loss in {i+1}th epoch is {loss_cal.item()}")



Loss in 1th epoch is 0.652819037437439
Loss in 11th epoch is 0.44244620203971863
Loss in 21th epoch is 0.2965417802333832
Loss in 31th epoch is 0.22076210379600525
Loss in 41th epoch is 0.17876215279102325


In [76]:
# model evaluation
with torch.no_grad():
  y_pred = model.forward(X_test_tensor)
  y_pred = (y_pred > 0.1).float()
  accuracy = (y_pred == y_test_tensor).float().mean()
  print(f'Accuracy: {accuracy.item()}')

Accuracy: 0.5067713260650635
